In [ ]:
# CELL 1 — Imports, Reproducibility, Setup
import sys
import os
import numpy as np
import random
import tensorflow as tf

# Force CPU only
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Reproducibility settings applied (CPU mode).")

# General imports
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import re
from math import atan2, asin
from scipy.signal import butter, filtfilt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from tensorflow import keras
from tensorflow.keras import layers

# Dataset paths
RAW_ROOT = Path("wide_data")
CLEAN_ROOT = Path("cleaned_data_lstm")
CLEAN_ROOT.mkdir(exist_ok=True)

# Quick sanity checks
print("Local dataset paths set.")
print("RAW_ROOT  =", RAW_ROOT.resolve())
print("CLEAN_ROOT=", CLEAN_ROOT.resolve())

if not RAW_ROOT.exists():
    raise FileNotFoundError(
        f"wide_data folder not found at: {RAW_ROOT.resolve()}\n"
        "Put wide_data in the same directory as the notebook, or adjust RAW_ROOT."
    )

print("Dataset ready (local).")

In [ ]:
# CELL 2 — Sensor Feature Definitions + Quaternion Math
SENSORS = ["L5", "T4", "C7", "T12"]

ACC_FEATURES  = ["Acceleration X(g)", "Acceleration Y(g)", "Acceleration Z(g)"]
GYRO_FEATURES = ["Angular velocity X(°/s)", "Angular velocity Y(°/s)", "Angular velocity Z(°/s)"]
QUAT_FEATURES = ["Quaternions 0()", "Quaternions 1()", "Quaternions 2()", "Quaternions 3()"]

MAG_WEAK_PREFIX = "Magnetic field"

POSTURES = sorted([p.name for p in RAW_ROOT.iterdir() if p.is_dir()])
LABEL_TO_ID = {p: i for i, p in enumerate(POSTURES)}

print("Detected Postures:", POSTURES)
print("Label Map:", LABEL_TO_ID)


# -------------------------
# Magnetometer detection
# -------------------------
def detect_mag_cols(df, sensor):
    cols = df.columns
    out = []

    for axis in ["x", "y", "z"]:
        matches = [
            c for c in cols
            if c.lower().startswith(sensor.lower() + "_magnetic field")
            and axis in c.lower()
        ]
        if not matches:
            raise KeyError(f" Missing magnetometer axis={axis} for sensor={sensor}\n{list(cols)}")

        out.append(matches[0])

    return out


# -------------------------
# Quaternion helpers
# -------------------------
def quat_reorder_to_wxyz(qx, qy, qz, qw):
    return np.array([qw, qx, qy, qz], dtype=np.float32)

def quat_conjugate(q):
    w,x,y,z = q
    return np.array([w, -x, -y, -z], dtype=np.float32)

def quat_multiply(q1, q2):
    w1,x1,y1,z1 = q1
    w2,x2,y2,z2 = q2
    return np.array([
        w1*w2 - x1*x2 - y1*y2 - z1*z2,
        w1*x2 + x1*w2 + y1*z2 - z1*y2,
        w1*y2 - x1*z2 + y1*w2 + z1*x2,
        w1*z2 + x1*y2 - y1*x2 + z1*w2
    ], dtype=np.float32)

def rotate_vector_by_quaternion(q, v):
    vq = np.array([0, v[0], v[1], v[2]], dtype=np.float32)
    return quat_multiply(quat_multiply(q, vq), quat_conjugate(q))[1:]

def quaternion_to_euler(q):
    w,x,y,z = q
    yaw   = atan2(2*(w*z + x*y), 1 - 2*(y*y + z*z))
    pitch = asin(np.clip(2*(w*y - z*x), -1, 1))
    roll  = atan2(2*(w*x + y*z), 1 - 2*(x*x + y*y))
    return yaw, pitch, roll

In [ ]:
# CELL 3 — Preprocessing Filters
def hampel_filter(x, window_size=5, n_sigmas=3):
    x = x.copy()
    y = x.copy()
    N = len(x)
    k = window_size

    for i in range(k, N-k):
        window = x[i-k:i+k+1]
        median = np.nanmedian(window)
        mad = np.nanmedian(np.abs(window - median))

        if mad < 1e-6:
            continue

        if abs(x[i] - median) > n_sigmas * 1.4826 * mad:
            y[i] = median

    return y

def butter_lowpass_filter(x, cutoff=3.0, fs=50, order=2):
    b,a = butter(order, cutoff/(0.5*fs), btype="low")
    try:
        return filtfilt(b,a,x,axis=0)
    except:
        return x

def remove_bias(x):
    return x - np.nanmean(x, axis=0)

In [ ]:
# CELL 4 — Full Preprocessing Pipeline
def preprocess_single_file(raw_path):
    """
    Preprocess a single raw CSV file and return the processed array (ALL FEATURES).
    Does NOT write to disk.
    """
    df = pd.read_csv(raw_path)

    # Keep relevant columns only
    keep = []
    for col in df.columns:
        for s in SENSORS:
            if col.startswith(f"{s}_Acceleration"):     keep.append(col)
            elif col.startswith(f"{s}_Angular velocity"): keep.append(col)
            elif col.startswith(f"{s}_Quaternions"):     keep.append(col)
            elif col.startswith(f"{s}_Magnetic field"):  keep.append(col)

    df = df[keep].copy().apply(pd.to_numeric, errors="coerce")

    df.replace([np.inf,-np.inf], np.nan, inplace=True)
    df.interpolate(limit_direction="both", inplace=True)
    df.ffill(inplace=True)
    df.bfill(inplace=True)

    blocks = []

    for sensor in SENSORS:

        acc = df[[f"{sensor}_{f}" for f in ACC_FEATURES]].values
        gyr = df[[f"{sensor}_{f}" for f in GYRO_FEATURES]].values
        quat_raw = df[[f"{sensor}_{f}" for f in QUAT_FEATURES]].values
        mag_cols = detect_mag_cols(df, sensor)
        mag = df[mag_cols].values

        # Quaternion normalization
        quats = []
        for (qx,qy,qz,qw) in quat_raw:
            q = quat_reorder_to_wxyz(qx,qy,qz,qw)
            q = q / np.linalg.norm(q)
            quats.append(q)
        quats = np.array(quats)

        # Remove bias
        acc = remove_bias(acc)
        gyr = remove_bias(gyr)
        mag = remove_bias(mag)

        # Hampel filtering
        for i in range(acc.shape[1]): acc[:,i] = hampel_filter(acc[:,i])
        for i in range(gyr.shape[1]): gyr[:,i] = hampel_filter(gyr[:,i])
        for i in range(mag.shape[1]): mag[:,i] = hampel_filter(mag[:,i])

        # Low-pass filter
        acc = butter_lowpass_filter(acc)
        gyr = butter_lowpass_filter(gyr)
        mag = butter_lowpass_filter(mag)

        # Orientation alignment + Euler
        A,G,M,E = [],[],[],[]

        for t in range(len(df)):
            q = quats[t]
            A.append(rotate_vector_by_quaternion(q, acc[t]))
            G.append(rotate_vector_by_quaternion(q, gyr[t]))
            M.append(rotate_vector_by_quaternion(q, mag[t]))
            E.append(quaternion_to_euler(q))

        A = np.array(A); G = np.array(G); M = np.array(M); E = np.array(E)

        A_mag = np.linalg.norm(A,axis=1,keepdims=True)
        G_mag = np.linalg.norm(G,axis=1,keepdims=True)
        M_mag = np.linalg.norm(M,axis=1,keepdims=True)

        block = np.concatenate([A,G,M,A_mag,G_mag,M_mag,E], axis=1)
        blocks.append(block)

    X_out = np.concatenate(blocks, axis=1)

    return X_out

In [ ]:
# CELL 5 — Load and Preprocess All Data
print("\n===== PREPROCESSING ALL FILES =====\n")

SUBJECT_RE = re.compile(r"Sub_(\d+)_Trial_(\d+)\.csv")

rows = []

for posture_dir in RAW_ROOT.iterdir():
    if not posture_dir.is_dir():
        continue

    files = list(posture_dir.glob("*.csv"))
    print(f"Processing posture: {posture_dir.name} ({len(files)} files)")

    for raw_file in files:
        # Preprocess in-memory (no disk I/O for cleaned files)
        X_processed = preprocess_single_file(raw_file)
        
        # Extract subject and trial info
        m = SUBJECT_RE.search(raw_file.name)
        if m:
            rows.append({
                "data": X_processed,
                "posture": posture_dir.name,
                "label": LABEL_TO_ID[posture_dir.name],
                "subject": int(m.group(1)),
                "trial": int(m.group(2)),
                "filename": raw_file.name
            })

df_items = pd.DataFrame(rows)

print(f"\nTotal processed files: {len(df_items)}")
print(f"Unique subjects: {df_items['subject'].nunique()}")
print(f"Feature dimension: {df_items['data'].iloc[0].shape[1]}")

In [ ]:
# CELL 6 — Windowing
FS = 50
WIN_LEN = 200
STRIDE = 100

print(f"\nWindow length = {WIN_LEN} frames ({WIN_LEN/FS:.2f} sec)")
print(f"Stride = {STRIDE} frames")

def make_windows(X):
    """
    Convert a full sequence (T,F) into windows (num_windows, WIN_LEN, F)
    with overlap.
    """
    T, F = X.shape

    # If trial is shorter than window, pad with zeros
    if T < WIN_LEN:
        w = np.zeros((WIN_LEN, F))
        w[:T] = X
        return w[None,:,:]

    windows = []
    for i in range(0, T - WIN_LEN + 1, STRIDE):
        windows.append(X[i:i+WIN_LEN])

    return np.array(windows)

In [ ]:
# CELL 7 — Create Window Dataset (ACC + EULER ONLY)
X_all = []
y_all = []
subj_all = []
trial_all = []

window_counter = 0

def select_acc_euler(X):
    """
    Input: (T, 60)
    Output: (T, 24)  -> 4 sensors × (acc(3) + euler(3))
    """

    selected = []

    for i in range(4):

        start = i * 15

        acc   = X[:, start + 0 : start + 3]
        euler = X[:, start + 12 : start + 15]

        selected.append(acc)
        selected.append(euler)

    return np.concatenate(selected, axis=1)


for _, row in df_items.iterrows():

    X = row["data"]  # Use preprocessed data from memory

    # keep only acc + euler
    X = select_acc_euler(X)

    windows = make_windows(X)

    for w in windows:

        X_all.append(w)
        y_all.append(row["label"])
        subj_all.append(row["subject"])
        trial_all.append(row["trial"])

    window_counter += len(windows)

X_all = np.array(X_all)
y_all = np.array(y_all)
subj_all = np.array(subj_all)
trial_all = np.array(trial_all)

print("\n===== DATASET SUMMARY =====")
print("Total windows:", window_counter)
print("X shape:", X_all.shape)
print("Features per timestep:", X_all.shape[2])

In [ ]:
# CELL 8 — Subject Split
EXCLUDED_SUBJECTS = [29]

subjects = sorted(
    s for s in np.unique(subj_all)
    if s not in EXCLUDED_SUBJECTS
)

print("Remaining subjects:", subjects)

train_subj, temp_subj = train_test_split(
    subjects,
    test_size=0.30,
    random_state=SEED
)

val_subj, test_subj = train_test_split(
    temp_subj,
    test_size=0.50,
    random_state=SEED
)

print("Train subjects:", train_subj)
print("Val subjects:", val_subj)
print("Test subjects:", test_subj)

# =========================================
# CREATE MASKS
# =========================================

train_mask = np.isin(subj_all, train_subj)
val_mask   = np.isin(subj_all, val_subj)
test_mask  = np.isin(subj_all, test_subj)

# =========================================
# SPLIT DATA
# =========================================

X_train = X_all[train_mask]
X_val   = X_all[val_mask]
X_test  = X_all[test_mask]

y_train = y_all[train_mask]
y_val   = y_all[val_mask]
y_test  = y_all[test_mask]

print("\nDataset shapes")
print("Train:", X_train.shape)
print("Val:", X_val.shape)
print("Test:", X_test.shape)

# =========================================
# ONE HOT ENCODE LABELS
# =========================================

y_train = keras.utils.to_categorical(y_train, num_classes=len(POSTURES))
y_val   = keras.utils.to_categorical(y_val,   num_classes=len(POSTURES))
y_test  = keras.utils.to_categorical(y_test,  num_classes=len(POSTURES))

print("\nLabel shapes")
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

In [ ]:
# CELL 9 — Per Sensor Normalization
SENSOR_SLICES = [
    slice(0,6),
    slice(6,12),
    slice(12,18),
    slice(18,24)
]

def compute_sensor_norm(X):

    means = []
    stds  = []

    for s in SENSOR_SLICES:

        flat = X[:,:,s].reshape(-1,6)

        mean = flat.mean(axis=0, keepdims=True)
        std  = flat.std(axis=0, keepdims=True) + 1e-8

        means.append(mean)
        stds.append(std)

    return means, stds


def apply_sensor_norm(X, means, stds):

    Xn = X.copy()

    for s, mean, std in zip(SENSOR_SLICES, means, stds):

        Xn[:,:,s] = (X[:,:,s] - mean) / std

    return Xn


means, stds = compute_sensor_norm(X_train)

X_train = apply_sensor_norm(X_train, means, stds)
X_val   = apply_sensor_norm(X_val, means, stds)
X_test  = apply_sensor_norm(X_test, means, stds)

print("Normalization applied")
print("Train shape:", X_train.shape)
print("Val shape:", X_val.shape)
print("Test shape:", X_test.shape)

In [ ]:
# CELL 10 — SensorDropout Layer
class SensorDropout(layers.Layer):

    def __init__(self, drop_prob=0.2):
        super().__init__()
        self.drop_prob = drop_prob

    def call(self, x, training=None):

        if not training:
            return x

        batch = tf.shape(x)[0]

        mask = tf.ones((batch, 4, 1), dtype=x.dtype)

        drop = tf.cast(
            tf.random.uniform((batch, 4, 1)) > self.drop_prob,
            x.dtype
        )

        mask = mask * drop

        mask = tf.repeat(mask, repeats=6, axis=2)

        mask = tf.reshape(mask, (batch, 1, 24))

        return x * mask

In [ ]:
# CELL 11 — Model with Sensor Attention + Global Avg Pool
def build_model(input_shape):

    inp = keras.Input(shape=input_shape)

    print("Model input:", input_shape)

    x = inp   

    # ---------------------------------
    # Split sensors
    # ---------------------------------

    L5  = layers.Lambda(lambda x: x[:,:,0:6])(x)
    T4  = layers.Lambda(lambda x: x[:,:,6:12])(x)
    C7  = layers.Lambda(lambda x: x[:,:,12:18])(x)
    T12 = layers.Lambda(lambda x: x[:,:,18:24])(x)

    print("Sensor feature size:", 6)

    # ---------------------------------
    # Sensor CNN block
    # ---------------------------------

    def sensor_block(x):

        x = layers.Conv1D(
            filters=32,
            kernel_size=5,
            padding="same",
            activation="relu"
        )(x)

        x = layers.BatchNormalization()(x)

        x = layers.Conv1D(
            filters=32,
            kernel_size=3,
            padding="same",
            activation="relu"
        )(x)

        x = layers.BatchNormalization()(x)

        # Feature attention
        attention = layers.Dense(x.shape[-1], activation="tanh")(x)
        attention = layers.Softmax(axis=-1)(attention)

        x = layers.Multiply()([x, attention])

        return x

    L5  = sensor_block(L5)
    T4  = sensor_block(T4)
    C7  = sensor_block(C7)
    T12 = sensor_block(T12)

    # ---------------------------------
    # Stack sensors
    # ---------------------------------

    sensors = layers.Lambda(
        lambda x: tf.stack(x, axis=2)
    )([L5, T4, C7, T12])

    # ---------------------------------
    # Sensor attention
    # ---------------------------------

    attention = layers.Dense(1, activation="tanh")(sensors)
    attention = layers.Softmax(axis=2)(attention)

    sensors = layers.Multiply()([sensors, attention])

    fused = layers.Lambda(
        lambda x: tf.reduce_sum(x, axis=2)
    )(sensors)

    # ---------------------------------
    # TEMPORAL POOLING
    # ---------------------------------

    x = layers.GlobalAveragePooling1D()(fused)

    # ---------------------------------
    # CLASSIFIER
    # ---------------------------------

    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.4)(x)

    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.3)(x)

    out = layers.Dense(
        len(POSTURES),
        activation="softmax"
    )(x)

    model = keras.Model(inp, out)

    model.compile(
        optimizer=keras.optimizers.Adam(1e-4),
        loss=tf.keras.losses.CategoricalCrossentropy(
            label_smoothing=0.05
        ),
        metrics=["accuracy"]
    )

    model.summary()

    return model

In [ ]:
# CELL 12 — Learning Rate Scheduler
lr_scheduler = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [ ]:
# CELL 13 — Build and Train Model
model = build_model(X_train.shape[1:])

early = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

hist = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=60,
    batch_size=32,
    callbacks=[early, lr_scheduler],
    verbose=1
)

In [ ]:
# CELL 14 — Training Curves
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(hist.history["loss"], label="train")
plt.plot(hist.history["val_loss"], label="val")
plt.title("Loss")
plt.legend()

plt.subplot(1,2,2)
plt.plot(hist.history["accuracy"], label="train")
plt.plot(hist.history["val_accuracy"], label="val")
plt.title("Accuracy")
plt.legend()

plt.show()

In [ ]:
# CELL 15 — Window Level Evaluation
# =========================================
# WINDOW LEVEL EVALUATION
# =========================================

probs = model.predict(X_test, verbose=0)

y_pred = np.argmax(probs, axis=1)
y_true = np.argmax(y_test, axis=1)

window_acc = np.mean(y_pred == y_true)

print("\n===== WINDOW RESULTS =====")
print("Window accuracy:", window_acc)

print("\nClassification Report (Window)")

print(classification_report(
    y_true,
    y_pred,
    labels=np.arange(len(POSTURES)),
    target_names=POSTURES,
    zero_division=0
))

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=np.arange(len(POSTURES))
)

ConfusionMatrixDisplay(cm, display_labels=POSTURES).plot()

plt.title("Window Confusion Matrix")
plt.show()

In [ ]:
# CELL 16 — Trial Level Evaluation
from collections import defaultdict

# =========================================
# TRIAL LEVEL EVALUATION
# =========================================

trial_probs = defaultdict(list)
trial_true = {}

test_subjects = subj_all[test_mask]
test_trials = trial_all[test_mask]

for p,true,s,t in zip(probs, y_true, test_subjects, test_trials):

    key = (s,t)   # subject + trial

    trial_probs[key].append(p)
    trial_true[key] = true

trial_pred = []
trial_true_list = []
trial_subjects = []

for k in trial_probs:

    w = np.array(trial_probs[k])

    summed = np.sum(np.log(w + 1e-8), axis=0)

    pred = np.argmax(summed)

    trial_pred.append(pred)
    trial_true_list.append(trial_true[k])
    trial_subjects.append(k[0])   # store subject id

trial_pred = np.array(trial_pred)
trial_true_list = np.array(trial_true_list)
trial_subjects = np.array(trial_subjects)

trial_acc = np.mean(trial_pred == trial_true_list)

print("\n===== TRIAL RESULTS =====")
print("Trial accuracy:", trial_acc)

print("\nClassification Report (Trial)")

print(classification_report(
    trial_true_list,
    trial_pred,
    labels=np.arange(len(POSTURES)),
    target_names=POSTURES,
    zero_division=0
))

cm = confusion_matrix(
    trial_true_list,
    trial_pred,
    labels=np.arange(len(POSTURES))
)

ConfusionMatrixDisplay(cm, display_labels=POSTURES).plot()

plt.title("Trial Confusion Matrix")
plt.show()

In [ ]:
# CELL 17 — Subject-Posture Level Evaluation
# =========================================
# SUBJECT-POSTURE LEVEL EVALUATION
# =========================================

subject_posture_votes = defaultdict(list)
subject_posture_true = {}

for pred,true,s in zip(trial_pred, trial_true_list, trial_subjects):

    key = (s,true)   # subject + posture

    subject_posture_votes[key].append(pred)
    subject_posture_true[key] = true

subject_pred = []
subject_true_list = []

for k in subject_posture_votes:

    votes = np.array(subject_posture_votes[k])

    pred = np.bincount(votes).argmax()

    subject_pred.append(pred)
    subject_true_list.append(subject_posture_true[k])

subject_pred = np.array(subject_pred)
subject_true_list = np.array(subject_true_list)

subject_acc = np.mean(subject_pred == subject_true_list)

print("\n===== SUBJECT-POSTURE RESULTS =====")
print("Subject/Posture accuracy:", subject_acc)

print("\nClassification Report (Subject/Posture)")

print(classification_report(
    subject_true_list,
    subject_pred,
    labels=np.arange(len(POSTURES)),
    target_names=POSTURES,
    zero_division=0
))

cm = confusion_matrix(
    subject_true_list,
    subject_pred,
    labels=np.arange(len(POSTURES))
)

ConfusionMatrixDisplay(cm, display_labels=POSTURES).plot()

plt.title("Subject/Posture Confusion Matrix")
plt.show()